# Задание 1. 

In [1]:
import pandas as pd

file = 'Тестовое_задание_Data_аналитик_ЦО_2025_2(1).xlsx'

products = pd.read_excel(file, sheet_name='Данные для задачи 1')
rules = pd.read_excel(file, sheet_name='Справочник наценок')

print('Товаров:', products.shape[0])
print('Правил наценки:', rules.shape[0])
products.head()

Товаров: 683
Правил наценки: 123


,Бренд,Код товара,Наименование товара,Категория1,Категория2,Категория3,Категория4,Сегмент,Цена закупки товара,letu.ru,iledebeaute.ru,rivegauche.ru,Ozon,Wildberries,MagnitMarket,Megamarket,lamoda.ru,tsum.ru,visagehall.ru
0,Clarins,19000412437,CLARINS Тушь д/ресниц Wonder volume xxl 8мл,Макияж,Глаза,Тушь для ресниц,NaN,Люкс,983,NaN,1029.0,NaN,NaN,NaN,NaN,NaN,1117.0,NaN,NaN
1,Clarins,14037100013,Clarins Флюид тональный устойчивый с омолажива...,Макияж,Лицо,Тональные основы,Жидкие тональные основы,Люкс,1957,NaN,2107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Clarins,19000033387,CLARINS Масло-блеск для губ Lip Comfort Oil т....,Макияж,Губы,Масло для губ_макияж,NaN,Люкс,850,NaN,1065.0,919.0,964.0,NaN,NaN,NaN,924.0,1574.0,964.0
3,Clarins,14037100015,Clarins Флюид тональный устойчивый с омолажива...,Макияж,Лицо,Тональные основы,Жидкие тональные основы,Люкс,2145,NaN,2355.0,3536.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Clarins,19000168458,Clarins Масло-блеск д/губ Lip Comfort 17 7мл,Макияж,Губы,Масло для губ_макияж,NaN,Люкс,835,NaN,924.0,1039.0,1045.0,NaN,NaN,NaN,1065.0,1742.0,NaN


## 1. В расчёт включаем правила со статусом «Да». 

In [2]:
rules = rules[rules['Активный'] == 'Да']

print('Активных правил:', rules.shape[0])
rules.head()

Активных правил: 120


,Сегмент рынка,Категория,Бренд,Продукт,Дата начала,Дата окончания,Наценка,Активный
0,Люкс,NaN,NaN,NaN,2025-01-01,NaT,1.2,Да
1,Массмаркет,NaN,NaN,NaN,2025-01-01,NaT,0.7,Да
2,Люкс,Макияж,NaN,NaN,2025-01-01,NaT,0.9,Да
3,Люкс,Уход,NaN,NaN,2025-01-01,NaT,1.5,Да
4,Массмаркет,Макияж,NaN,NaN,2025-01-01,NaT,0.5,Да


## 2. Распределем по приоритетам - Продукт

In [3]:
# Продукт.
product_rules = rules[~rules['Продукт'].isnull()][['Продукт', 'Наценка']]

prices = pd.merge(products, product_rules, how='left', left_on='Код товара', right_on='Продукт')
prices.drop(columns=['Продукт'], inplace=True)

prices['Правило'] = prices['Наценка'].isnull().apply(
    lambda missing: None if missing else 'Продукт'
)

prices[['Код товара', 'Наценка', 'Правило']].head()

,Код товара,Наценка,Правило
0,19000412437,NaN,None
1,14037100013,NaN,None
2,19000033387,NaN,None
3,14037100015,NaN,None
4,19000168458,NaN,None


## 3. По бренду и категории

Проверяем категории по убыванию приоритета: 3 → 2 → 1.

In [4]:
# Бренд и категория 
brand_category_rules = rules[(~rules['Бренд'].isnull()) &(~rules['Категория'].isnull()) &(rules['Продукт'].isnull())
    ][['Бренд', 'Категория', 'Наценка']].rename(columns={'Наценка': 'Новая наценка'})

for category in ['Категория3', 'Категория2', 'Категория1']:
    prices = pd.merge( prices, brand_category_rules, how='left', left_on=['Бренд', category], right_on=['Бренд', 'Категория'])

    prices['Правило'] = prices['Правило'].fillna(
        prices['Новая наценка'].isnull().apply(
            lambda missing: None if missing else 'Бренд / ' + category
        )
    )
    prices['Наценка'] = prices['Наценка'].fillna(prices['Новая наценка'])
    prices.drop(columns=['Категория', 'Новая наценка'], inplace=True)

prices[['Код товара', 'Бренд', 'Категория3', 'Наценка', 'Правило']].head()

,Код товара,Бренд,Категория3,Наценка,Правило
0,19000412437,Clarins,Тушь для ресниц,NaN,None
1,14037100013,Clarins,Тональные основы,NaN,None
2,19000033387,Clarins,Масло для губ_макияж,NaN,None
3,14037100015,Clarins,Тональные основы,NaN,None
4,19000168458,Clarins,Масло для губ_макияж,NaN,None


## 4. Наценка по бренду

Общие правила бренда применяются только к тем товарам,  что ранее остались без наценки


In [5]:
brand_rules = rules[(~rules['Бренд'].isnull()) &(rules['Категория'].isnull()) &(rules['Продукт'].isnull())
][['Бренд', 'Наценка']].rename(columns={'Наценка': 'Новая наценка'})

prices = pd.merge(prices, brand_rules, how='left', on='Бренд')
prices['Правило'] = prices['Правило'].fillna(
    prices['Новая наценка'].isnull().apply(lambda missing: None if missing else 'Бренд')
)
prices['Наценка'] = prices['Наценка'].fillna(prices['Новая наценка'])
prices.drop(columns=['Новая наценка'], inplace=True)

prices[['Код товара', 'Наценка', 'Правило']].head()

,Код товара,Наценка,Правило
0,19000412437,1.03,Бренд
1,14037100013,1.03,Бренд
2,19000033387,1.03,Бренд
3,14037100015,1.03,Бренд
4,19000168458,1.03,Бренд


## 5. По сегменту и категории
Для товаров без правила по товару или бренду проверяем сочетания сегмента с категориями 3, 2 и 1.

In [6]:
segment_category_rules = rules[(~rules['Сегмент рынка'].isnull()) & (~rules['Категория'].isnull()) &
    (rules['Бренд'].isnull()) & (rules['Продукт'].isnull()) ][['Сегмент рынка', 'Категория', 'Наценка']].rename(
    columns={'Наценка': 'Новая наценка'})

for category in ['Категория3', 'Категория2', 'Категория1']:
    prices = pd.merge(prices, segment_category_rules, how='left', left_on=['Сегмент', category],
                      right_on=['Сегмент рынка', 'Категория']
    )

    prices['Правило'] = prices['Правило'].fillna(
        prices['Новая наценка'].isnull().apply(
            lambda missing: None if missing else 'Сегмент / ' + category
        )
    )
    prices['Наценка'] = prices['Наценка'].fillna(prices['Новая наценка'])
    prices.drop(columns=['Сегмент рынка', 'Категория', 'Новая наценка'], inplace=True)

prices[['Код товара', 'Сегмент', 'Наценка', 'Правило']].head()

,Код товара,Сегмент,Наценка,Правило
0,19000412437,Люкс,1.03,Бренд
1,14037100013,Люкс,1.03,Бренд
2,19000033387,Люкс,1.03,Бренд
3,14037100015,Люкс,1.03,Бренд
4,19000168458,Люкс,1.03,Бренд


## 6. Сегмент

Последний уровень приоритета. После него проверяем, что наценка определена для каждого товара.

In [7]:
segment_rules = rules[(~rules['Сегмент рынка'].isnull()) & (rules['Категория'].isnull()) & (rules['Бренд'].isnull()) &
    (rules['Продукт'].isnull())][['Сегмент рынка', 'Наценка']].rename(columns={'Наценка': 'Новая наценка'})

prices = pd.merge( prices, segment_rules, how='left',left_on='Сегмент', right_on='Сегмент рынка')

prices['Правило'] = prices['Правило'].fillna(
    prices['Новая наценка'].isnull().apply(lambda missing: None if missing else 'Сегмент')
)
prices['Наценка'] = prices['Наценка'].fillna(prices['Новая наценка'])
prices.drop(columns=['Сегмент рынка', 'Новая наценка'], inplace=True)

print('Строк после объединений:', prices.shape[0])
print('Товаров без наценки:', prices['Наценка'].isnull().agg('sum'))
prices.groupby('Правило').agg(Товаров=('Код товара', 'count')).reset_index()

Строк после объединений: 683
Товаров без наценки: 0


,Правило,Товаров
0,Бренд,380
1,Бренд / Категория2,26
2,Бренд / Категория3,273
3,Продукт,4


## 7. Цена до скидки и индексы конкурентов

Цена до скидки рассчитывается по закупочной стоимости и применённой наценке. Индекс цены — отношение нашей цены к цене конкурента; значения выше 1 показывают, что наша цена выше.

In [8]:
prices['Наша цена без скидки'] = (
    prices['Цена закупки товара'] * (1 + prices['Наценка'])
)

competitors = [
    'letu.ru', 'iledebeaute.ru', 'rivegauche.ru', 'Ozon', 'Wildberries',
    'MagnitMarket', 'Megamarket', 'lamoda.ru', 'tsum.ru', 'visagehall.ru'
]

# Индекс рассчитываем отдельно для каждого конкурент.
for competitor in competitors:
    prices['Индекс ' + competitor] = (
        prices['Наша цена без скидки'] / prices[competitor]
    )

index_cols = ['Индекс ' + competitor for competitor in competitors]

# Цены всех конкурентов
prices[['Код товара', 'Наша цена без скидки'] + competitors].head(10)

,Код товара,Наша цена без скидки,letu.ru,iledebeaute.ru,rivegauche.ru,Ozon,Wildberries,MagnitMarket,Megamarket,lamoda.ru,tsum.ru,visagehall.ru
0,19000412437,1995.49,NaN,1029.0,NaN,NaN,NaN,NaN,NaN,1117.0,NaN,NaN
1,14037100013,3972.71,NaN,2107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19000033387,1725.50,NaN,1065.0,919.0,964.0,NaN,NaN,NaN,924.0,1574.0,964.0
3,14037100015,4354.35,NaN,2355.0,3536.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19000168458,1695.05,NaN,924.0,1039.0,1045.0,NaN,NaN,NaN,1065.0,1742.0,NaN
5,19000430971,1822.94,NaN,NaN,999.0,1065.0,NaN,NaN,NaN,NaN,NaN,1085.0
6,19000283309,1995.49,NaN,1094.0,1146.0,1208.0,NaN,NaN,NaN,1117.0,2052.0,1185.0
7,19000347411,2814.00,NaN,NaN,1495.0,1687.0,NaN,NaN,NaN,NaN,NaN,NaN
8,14037100011,7019.74,NaN,3904.0,6104.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,19000033383,1950.83,NaN,1185.0,1159.0,1085.0,NaN,NaN,NaN,1145.0,1943.0,1145.0


In [9]:
# Индексы по всем конкурентам.
prices[['Код товара'] + index_cols].head(10)

,Код товара,Индекс letu.ru,Индекс iledebeaute.ru,Индекс rivegauche.ru,Индекс Ozon,Индекс Wildberries,Индекс MagnitMarket,Индекс Megamarket,Индекс lamoda.ru,Индекс tsum.ru,Индекс visagehall.ru
0,19000412437,NaN,1.939252,NaN,NaN,NaN,NaN,NaN,1.786473,NaN,NaN
1,14037100013,NaN,1.885482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19000033387,NaN,1.620188,1.877584,1.789938,NaN,NaN,NaN,1.867424,1.096252,1.789938
3,14037100015,NaN,1.848981,1.231434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19000168458,NaN,1.834470,1.631424,1.622057,NaN,NaN,NaN,1.591596,0.973048,NaN
5,19000430971,NaN,NaN,1.824765,1.711681,NaN,NaN,NaN,NaN,NaN,1.680129
6,19000283309,NaN,1.824031,1.741265,1.651896,NaN,NaN,NaN,1.786473,0.972461,1.683958
7,19000347411,NaN,NaN,1.882274,1.668050,NaN,NaN,NaN,NaN,NaN,NaN
8,14037100011,NaN,1.798089,1.150023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,19000033383,NaN,1.646270,1.683201,1.798000,NaN,NaN,NaN,1.703782,1.004030,1.703782


## 8. Предложенная цена и скидка

Необходимо найти  минимальную конкурентную цену. 
Нижняя граница определяется маржинальностью 10%: Цена закупки / 0.9. 
Если рынок дешевле допустимой границы, предлагаем цену по нижней границе.

In [10]:
# Минимальная цена среди конкурентов с доступными данными.
prices['Минимальная цена рынка'] = prices[competitors].agg('min', axis=1)
prices['Нижняя граница цены'] = prices['Цена закупки товара'] / 0.90

prices['Можно сравняться с рынком'] = (
    prices['Минимальная цена рынка'] >= prices['Нижняя граница цены']
)

# Сначала выбираем меньшую цену.
# Если мониторинг пустой, остаётся наша цена.
prices['Предложенная цена'] = prices[
    ['Наша цена без скидки', 'Минимальная цена рынка']
].agg('min', axis=1)

# Не снижаем цену ниже уровня маржинальности 10%.
prices['Предложенная цена'] = prices[
    ['Предложенная цена', 'Нижняя граница цены']
].agg('max', axis=1)

prices['Скидка %'] = 100 * (
    1 - prices['Предложенная цена'] / prices['Наша цена без скидки']
)
prices['Маржинальность %'] = 100 * (
    1 - prices['Цена закупки товара'] / prices['Предложенная цена']
)

prices[['Код товара', 'Цена закупки товара', 'Наша цена без скидки',
        'Минимальная цена рынка', 'Предложенная цена', 'Скидка %',
        'Маржинальность %', 'Можно сравняться с рынком']].head(12)

,Код товара,Цена закупки товара,Наша цена без скидки,Минимальная цена рынка,Предложенная цена,Скидка %,Маржинальность %,Можно сравняться с рынком
0,19000412437,983,1995.49,1029.0,1092.222222,45.265463,10.000000,False
1,14037100013,1957,3972.71,2107.0,2174.444444,45.265463,10.000000,False
2,19000033387,850,1725.50,919.0,944.444444,45.265463,10.000000,False
3,14037100015,2145,4354.35,2355.0,2383.333333,45.265463,10.000000,False
4,19000168458,835,1695.05,924.0,927.777778,45.265463,10.000000,False
5,19000430971,898,1822.94,999.0,999.000000,45.198416,10.110110,True
6,19000283309,983,1995.49,1094.0,1094.000000,45.176373,10.146252,True
7,19000347411,1340,2814.00,1495.0,1495.000000,46.872779,10.367893,True
8,14037100011,3458,7019.74,3904.0,3904.000000,44.385405,11.424180,True
9,19000033383,961,1950.83,1085.0,1085.000000,44.382647,11.428571,True


## 9. Проверка результатов

Проверяем соблюдение ограничения по маржинальности и количество товаров, по которым минимальная цена рынка недостижима.

In [11]:
print('Всего товаров:', prices.shape[0])
print('Товаров без наценки:', prices['Наценка'].isnull().agg('sum'))
print('Маржинальность ниже 10%:', (prices['Маржинальность %'] < 10 - 0.000001).agg('sum'))
print('Нельзя сравняться с минимумом рынка:', (prices['Можно сравняться с рынком'] == False).agg('sum'))

prices.groupby('Можно сравняться с рынком').agg(
    Товаров=('Код товара', 'count'),
    Средняя_скидка=('Скидка %', 'mean')
).reset_index()

Всего товаров: 683
Товаров без наценки: 0
Маржинальность ниже 10%: 0
Нельзя сравняться с минимумом рынка: 156


,Можно сравняться с рынком,Товаров,Средняя_скидка
0,False,156,30.219783
1,True,527,19.985088


In [12]:
# Товары с наибольшей предлагаемой скидкой.
prices.sort_values('Скидка %', ascending=False)[
    ['Бренд', 'Код товара', 'Правило', 'Наценка', 'Наша цена без скидки',
     'Минимальная цена рынка', 'Предложенная цена', 'Скидка %', 'Маржинальность %']
].head(15)

,Бренд,Код товара,Правило,Наценка,Наша цена без скидки,Минимальная цена рынка,Предложенная цена,Скидка %,Маржинальность %
81,Clarins,19000359936,Продукт,2.50,8165.50,3168.0,3168.000000,61.202621,26.357323
104,Clarins,19000419067,Продукт,2.50,6429.50,2560.0,2560.000000,60.183529,28.242188
299,Clarins,19000217070,Продукт,2.50,4711.00,2199.0,2199.000000,53.322012,38.790359
133,Clarins,14035900003,Продукт,2.00,3216.00,1557.0,1557.000000,51.585821,31.149647
7,Clarins,19000347411,Бренд / Категория3,1.10,2814.00,1495.0,1495.000000,46.872779,10.367893
1,Clarins,14037100013,Бренд,1.03,3972.71,2107.0,2174.444444,45.265463,10.000000
0,Clarins,19000412437,Бренд,1.03,1995.49,1029.0,1092.222222,45.265463,10.000000
2,Clarins,19000033387,Бренд,1.03,1725.50,919.0,944.444444,45.265463,10.000000
4,Clarins,19000168458,Бренд,1.03,1695.05,924.0,927.777778,45.265463,10.000000
3,Clarins,14037100015,Бренд,1.03,4354.35,2355.0,2383.333333,45.265463,10.000000


## 10. Итоговая таблица

В `result` сохранены цены и индексы по всем десяти конкурентам, выбранное правило наценки, предложенная цена и рассчитанная скидка.

In [13]:
result = prices[
    ['Код товара', 'Бренд', 'Наименование товара', 'Цена закупки товара',
     'Наценка', 'Правило', 'Наша цена без скидки']
    + competitors
    + index_cols
    + ['Минимальная цена рынка', 'Нижняя граница цены',
       'Предложенная цена', 'Скидка %', 'Маржинальность %',
       'Можно сравняться с рынком']
]

print('Итоговая таблица:', result.shape[0], 'товаров,', result.shape[1], 'столбцов')
result.head()

Итоговая таблица: 683 товаров, 33 столбцов


,Код товара,Бренд,Наименование товара,Цена закупки товара,Наценка,Правило,Наша цена без скидки,letu.ru,iledebeaute.ru,rivegauche.ru,Ozon,Wildberries,MagnitMarket,Megamarket,lamoda.ru,tsum.ru,visagehall.ru,Индекс letu.ru,Индекс iledebeaute.ru,Индекс rivegauche.ru,Индекс Ozon,Индекс Wildberries,Индекс MagnitMarket,Индекс Megamarket,Индекс lamoda.ru,Индекс tsum.ru,Индекс visagehall.ru,Минимальная цена рынка,Нижняя граница цены,Предложенная цена,Скидка %,Маржинальность %,Можно сравняться с рынком
0,19000412437,Clarins,CLARINS Тушь д/ресниц Wonder volume xxl 8мл,983,1.03,Бренд,1995.49,NaN,1029.0,NaN,NaN,NaN,NaN,NaN,1117.0,NaN,NaN,NaN,1.939252,NaN,NaN,NaN,NaN,NaN,1.786473,NaN,NaN,1029.0,1092.222222,1092.222222,45.265463,10.0,False
1,14037100013,Clarins,Clarins Флюид тональный устойчивый с омолажива...,1957,1.03,Бренд,3972.71,NaN,2107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.885482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2107.0,2174.444444,2174.444444,45.265463,10.0,False
2,19000033387,Clarins,CLARINS Масло-блеск для губ Lip Comfort Oil т....,850,1.03,Бренд,1725.50,NaN,1065.0,919.0,964.0,NaN,NaN,NaN,924.0,1574.0,964.0,NaN,1.620188,1.877584,1.789938,NaN,NaN,NaN,1.867424,1.096252,1.789938,919.0,944.444444,944.444444,45.265463,10.0,False
3,14037100015,Clarins,Clarins Флюид тональный устойчивый с омолажива...,2145,1.03,Бренд,4354.35,NaN,2355.0,3536.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.848981,1.231434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2355.0,2383.333333,2383.333333,45.265463,10.0,False
4,19000168458,Clarins,Clarins Масло-блеск д/губ Lip Comfort 17 7мл,835,1.03,Бренд,1695.05,NaN,924.0,1039.0,1045.0,NaN,NaN,NaN,1065.0,1742.0,NaN,NaN,1.834470,1.631424,1.622057,NaN,NaN,NaN,1.591596,0.973048,NaN,924.0,927.777778,927.777778,45.265463,10.0,False


## Вывод

Для товаров, по которым минимальная конкурентная цена совместима с маржинальностью от 10%, рассчитана возможная скидка. Для остальных товаров снижение до минимума рынка нецелесообразно.

Перед применением цен необходимо согласовать правила округления и проверить актуальность цен мониторинга.